# P4 — Doublons, métrique, coût des erreurs

**Enchaînement : 1.7 Doublons → 1.11 La métrique → 1.12 Le coût des erreurs**

Phase 1 du projet *Santander Satisfaction* : connaître les données avant de modéliser.
Chaque tâche suit le même plan : **la question**, **la mesure** (le code), **le constat**, **la décision**
que le porteur doit pouvoir défendre seul.

Pour relancer : *Exécuter tout*. Les données sont lues dans `data/` (récupérées par `git pull`), tables entières.

In [1]:
# --- Réglages communs à toute l'équipe ------------------------------------
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

# Les données sont dans data/ à la racine du dépôt (récupérées par git pull)
DATA = next(p for p in (Path("data"), Path("../data")) if (p / "train.csv").is_file())
CIBLE, ID = "TARGET", "ID"
GRAINE = 42             # même graine pour toute l'équipe : même découpage
PART_VALIDATION = 0.20  # protocole : 80 % apprentissage / 20 % validation, stratifié
PLANCHER = 0.6890       # AUC de la règle métier, donnée par l'énoncé
SENTINELLES = (-999999, 9999999999)
FAMILLES = ("saldo", "imp", "num", "ind", "delta")

t0 = time.perf_counter()
train = pd.read_csv(DATA / "train.csv")
a_decider = pd.read_csv(DATA / "a_decider.csv")
DUREE_CHARGEMENT = time.perf_counter() - t0
VARIABLES = [c for c in train.columns if c not in (ID, CIBLE)]
y = train[CIBLE]


def famille(col):
    if col in (ID, CIBLE):
        return col
    debut = col.split("_")[0]
    return debut if debut in FAMILLES else "isolée"


def pct(x):
    return f"{100 * x:.2f} %"


print(f"train : {train.shape} · a_decider : {a_decider.shape} · lus en {DUREE_CHARGEMENT:.1f} s depuis {DATA}")

train : (60816, 371) · a_decider : (15204, 370) · lus en 0.8 s depuis ../data


## 1.7 — Doublons

**Question** : y a-t-il des clients en double, des dossiers identiques, des colonnes identiques ?

In [2]:
print("ID en double : train", int(train[ID].duplicated().sum()), "· a_decider", int(a_decider[ID].duplicated().sum()),
      "· communs aux deux tables", len(set(train[ID]) & set(a_decider[ID])))
double = train.duplicated(subset=VARIABLES, keep=False)
print(f"Lignes de train dont le dossier (hors ID) existe ailleurs dans train : {int(double.sum())}")
nb_cibles = train.groupby(VARIABLES, sort=False)[CIBLE].transform("nunique")
conflits = train[nb_cibles > 1]
print(f"Dont dossiers identiques de cibles opposées : {len(conflits)} lignes, {int(conflits[CIBLE].sum())} mécontents")
cles_train = set(pd.util.hash_pandas_object(train[VARIABLES], index=False))
cles_dec = pd.util.hash_pandas_object(a_decider[VARIABLES], index=False)
print(f"Clients de a_decider dont le dossier est identique à un dossier de train : {int(cles_dec.isin(cles_train).sum())}")

ID en double : train 0 · a_decider 0 · communs aux deux tables 0


Lignes de train dont le dossier (hors ID) existe ailleurs dans train : 4572


Dont dossiers identiques de cibles opposées : 1994 lignes, 256 mécontents


Clients de a_decider dont le dossier est identique à un dossier de train : 1130


In [3]:
# Colonnes identiques (on écarte les constantes, toutes identiques entre elles et traitées par P3)
tout = pd.concat([train[VARIABLES], a_decider[VARIABLES]])
non_constantes = [c for c in VARIABLES if tout[c].nunique() > 1]
empreintes = pd.Series({c: pd.util.hash_pandas_object(tout[c], index=False).sum() for c in non_constantes})
groupes = [list(g.index) for _, g in empreintes.groupby(empreintes) if len(g) > 1]
JUMELLES = [c for g in groupes for c in g[1:]]  # on garde la première de chaque groupe
print(f"{len(groupes)} groupes de colonnes identiques : {len(JUMELLES)} colonnes en trop")
pd.DataFrame({"gardée": [g[0] for g in groupes], "copies supprimées": [", ".join(g[1:]) for g in groupes]})

26 groupes de colonnes identiques : 41 colonnes en trop


,gardée,copies supprimées
0,num_var37_0,num_var37
1,num_reemb_var17_hace3,"num_reemb_var33_ult1, num_trasp_var33_out_ult1"
2,delta_imp_trasp_var17_in_1y3,delta_num_trasp_var17_in_1y3
3,num_var32_0,num_var32
4,num_var6,"num_var13_medio_0, num_var13_medio, num_var18_..."
5,ind_var20,ind_var7_recib_ult1
6,saldo_var13_medio,saldo_medio_var13_medio_ult1
7,ind_var26_0,ind_var26
8,delta_imp_amort_var18_1y3,delta_imp_amort_var34_1y3
9,delta_imp_reemb_var13_1y3,delta_num_reemb_var13_1y3


**Constat**

- **Aucun client en double** : chaque `ID` est unique, aucun n'est partagé entre les deux tables.
- Mais **4 572 dossiers** de train sont identiques à un autre (clients différents, mêmes valeurs) — fréquent quand presque tout vaut 0. **1 994** de ces lignes ont des **cibles opposées** (256 mécontents) : même dossier, issue différente. Et 1 130 clients de `a_decider` ont un dossier identique à un client de train.
- **41 colonnes** sont des copies exactes d'une autre (26 groupes), par exemple `num_var32_0` = `num_var32`.

**Décision**

- **Garder toutes les lignes** : ce sont des clients distincts. Supprimer les doublons fausserait le taux de mécontents. Les dossiers de cibles opposées sont du bruit irréductible : l'AUC maximale est inférieure à 1.
- **Supprimer les 41 colonnes en double** (garder la première de chaque groupe) : elles n'apportent rien et ralentissent le pipeline.

## 1.11 — La métrique

**Question** : sur quel chiffre juge-t-on un modèle ? Pourquoi pas l'exactitude ?

In [4]:
# --- Protocole de validation et règle à une variable -------------------------
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 80 / 20 stratifié sur TARGET, graine fixe : chaque partie garde 3,96 % de mécontents
i_app, i_val = train_test_split(train.index, test_size=PART_VALIDATION, stratify=y, random_state=GRAINE)

# Chaque variable prise seule comme score. Le sens (grand = mécontent, ou l'inverse)
# est choisi sur l'apprentissage ; l'AUC est mesurée sur la validation, jamais vue.
lignes = []
for c in VARIABLES:
    if train.loc[i_app, c].nunique() < 2:
        continue
    sens = 1 if roc_auc_score(y[i_app], train.loc[i_app, c]) >= 0.5 else -1
    lignes.append((c, sens, roc_auc_score(y[i_val], sens * train.loc[i_val, c])))
regles = (pd.DataFrame(lignes, columns=["variable", "sens", "AUC validation"])
          .sort_values("AUC validation", ascending=False).set_index("variable"))

# Hypothèse : la règle de la banque est celle dont l'AUC est la plus proche du plancher
REGLE = (regles["AUC validation"] - PLANCHER).abs().idxmin()
SENS = int(regles.loc[REGLE, "sens"])
y_val = y[i_val].to_numpy()
score_val = SENS * train.loc[i_val, REGLE].to_numpy()
print(f"validation : {len(i_val)} clients, {int(y_val.sum())} mécontents · règle métier : {REGLE} "
      f"(AUC {regles.loc[REGLE, 'AUC validation']:.4f})")

validation : 12164 clients, 481 mécontents · règle métier : var15 (AUC 0.6903)


In [5]:
from sklearn.metrics import average_precision_score

ordre = np.argsort(-score_val, kind="stable")
lignes = [("Personne n'est mécontent", pct(1 - y_val.mean()), 0.5, y_val.mean(), "0 %"),
          (f"Règle métier ({REGLE})", "—", roc_auc_score(y_val, score_val), average_precision_score(y_val, score_val), "—")]
for budget in (0.05, 0.10, 0.20):
    n_appels = int(budget * len(y_val))
    lignes.append((f"Règle métier, on rappelle les {int(budget * 100)} % les plus à risque", "—", np.nan, np.nan,
                   pct(y_val[ordre[:n_appels]].sum() / y_val.sum())))
pd.DataFrame(lignes, columns=["règle", "exactitude", "AUC ROC", "AUC précision-rappel", "mécontents trouvés"]).set_index("règle")

,exactitude,AUC ROC,AUC précision-rappel,mécontents trouvés
règle,,,,
Personne n'est mécontent,96.05 %,0.5000,0.0395,0 %
Règle métier (var15),—,0.6903,0.0613,—
"Règle métier, on rappelle les 5 % les plus à risque",—,NaN,NaN,4.78 %
"Règle métier, on rappelle les 10 % les plus à risque",—,NaN,NaN,12.89 %
"Règle métier, on rappelle les 20 % les plus à risque",—,NaN,NaN,35.14 %


**Constat**

- Ne rappeler **personne** donne **96 %** d'exactitude et ne trouve **aucun** mécontent : **l'exactitude récompense le silence**.
- La règle métier (âge) : AUC ROC **0,69**. En rappelant les 10 % les plus âgés, on ne trouve que **13 %** des mécontents.

**Décision**

- **Métrique principale : AUC ROC** — la probabilité qu'un mécontent soit classé devant un satisfait. Elle juge le **classement**, ne dépend ni du seuil ni du déséquilibre, et c'est la métrique du plancher (0,6890) : on compare des chiffres comparables.
- **En contrôle : AUC précision-rappel** (niveau de base = 3,96 %) et **mécontents trouvés pour un budget d'appels** donné — c'est ce que la banque comprend.
- Le classement ne suffit pas pour décider qui rappeler : le **seuil** se fixe avec le coût des erreurs (1.12).

## 1.12 — Le coût des erreurs

**Question** : un mécontent qu'on n'a pas rappelé et un appel inutile ne coûtent pas la même chose. Combien rappeler ?

On note **r** = coût d'un mécontent raté / coût d'un appel inutile. La banque doit fixer r ; on calcule la meilleure décision pour plusieurs valeurs, avec la règle métier, sur la validation.

In [6]:
trouves = np.r_[0, np.cumsum(y_val[ordre])]   # mécontents trouvés en rappelant les k premiers
k = np.arange(len(y_val) + 1)
total = int(y_val.sum())
lignes = []
for r in (2, 5, 10, 25, 50):
    cout = (k - trouves) + r * (total - trouves)  # appels inutiles × 1 + mécontents ratés × r
    best = int(np.argmin(cout))
    lignes.append((r, round(1 / (1 + r), 3), best, pct(best / len(y_val)), int(trouves[best]), total - int(trouves[best]),
                   int(cout[best]), r * total, len(y_val) - total))
pd.DataFrame(lignes, columns=["r", "seuil 1/(1+r)", "appels", "part rappelée", "trouvés", "ratés",
                              "coût", "coût sans appel", "coût tout appeler"]).set_index("r")

,seuil 1/(1+r),appels,part rappelée,trouvés,ratés,coût,coût sans appel,coût tout appeler
r,,,,,,,,
2,0.3330,0,0.00 %,0,481,962,962,11683
5,0.1670,0,0.00 %,0,481,2405,2405,11683
10,0.0910,0,0.00 %,0,481,4810,4810,11683
25,0.0380,6070,49.90 %,401,80,7669,12025,11683
50,0.0200,7373,60.61 %,441,40,8932,24050,11683


**Constat**

- Avec la règle métier, **rappeler ne vaut le coup que si un mécontent raté coûte au moins ~25 fois un appel inutile**. En dessous (r = 2, 5, 10), la meilleure décision est de **ne rappeler personne** : la règle est trop faible pour que ses appels rapportent.
- À r = 25, on rappelle la moitié des clients pour trouver 401 mécontents sur 481.

**Décision**

- **Le rapport r est une donnée métier à demander à la banque.** Le seuil s'en déduit : avec des probabilités bien calibrées, on rappelle tout client dont le risque dépasse **1 / (1 + r)** (9 % pour r = 10).
- Si la banque fixe plutôt un **budget d'appels** (ex. 10 % des clients), le seuil est imposé par le budget : on rappelle les plus à risque jusqu'à épuisement.
- Un bon modèle doit faire **baisser le coût** à r égal : c'est la traduction en argent de l'AUC.